In [0]:
# Parameters
dbutils.widgets.text("file_name", "")
dbutils.widgets.text("zip_path", "")
dbutils.widgets.text("pipeline_name", "customer_billing")
dbutils.widgets.text("run_id", "")

file_name = dbutils.widgets.get("file_name")
zip_path = dbutils.widgets.get("zip_path")
pipeline_name = dbutils.widgets.get("pipeline_name")
run_id = dbutils.widgets.get("run_id")

print(f"file_name: [{file_name}]")
print(f"zip_path: [{zip_path}]")
print(f"pipeline_name: [{pipeline_name}]")
print(f"run_id: [{run_id}]")

if not file_name or not zip_path or not run_id:
    raise ValueError("file_name, zip_path and run_id are required")

# Paths to your existing Databricks notebooks
stage1_notebook = "/Workspace/Users/wade.amit8895@outlook.com/Azure-Databricks-Projects/customer_billing/notebooks/bronze/stage_1_unzip_files"

stage2_notebook = "/Workspace/Users/wade.amit8895@outlook.com/Azure-Databricks-Projects/customer_billing/notebooks/bronze/stage_2_load_data_to_bronze"

stage3_notebook = "/Workspace/Users/wade.amit8895@outlook.com/Azure-Databricks-Projects/customer_billing/notebooks/bronze/stage_3_archive_zip"

# Stage 1: Validate and extract ZIP
actual_unzip_path = dbutils.notebook.run(stage1_notebook, 0,
    {
        "file_name": file_name,
        "pipeline_name": pipeline_name,
        "zip_path": zip_path,
        "run_id": run_id
    }
)
print(f"Extraction folder: {actual_unzip_path}")

# Stage 2: Load data into Bronze
dbutils.notebook.run(stage2_notebook, 0,
    {
        "file_name": file_name,
        "pipeline_name": pipeline_name,
        "run_id": run_id,
        "unzip_path": actual_unzip_path
    }
)

# Stage 3: Archive ZIP
dbutils.notebook.run(stage3_notebook, 0,
    {
        "file_name": file_name,
        "pipeline_name": pipeline_name,
        "run_id": run_id,
        "unzip_path": actual_unzip_path
    }
)
print(f"ZIP processing completed: {file_name}")